# rnn_walk_forward_validation_tuning

Recurrent neural network validation tuning using the full-history session-aligned dataset.

This notebook uses compact GRU/LSTM sequence models. Instead of giving the model a single engineered row, it builds rolling ticker-specific sequences ending on the prediction date. The default grid is intentionally smaller than the SVM/RF/logistic grids because each candidate requires neural-network training.

The pipeline keeps the same evaluation discipline as the other notebooks: it splits on the full session calendar while keeping neutral targets as the middle class, selects feature/hyperparameter configurations with expanding-window walk-forward validation, scores the selected multiclass configuration on the holdout validation period, and evaluates the frozen model on the untouched test split.


In [ ]:
from __future__ import annotations

import gc
import os
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers
except ImportError as exc:
    tf = None
    keras = None
    layers = None
    regularizers = None
    TENSORFLOW_IMPORT_ERROR = exc
else:
    TENSORFLOW_IMPORT_ERROR = None

if tf is None:
    raise ImportError(
        "TensorFlow is required for this RNN notebook. Update the project environment from environment.yml "
        "or install into the active notebook kernel with: python -m pip install \"tensorflow>=2.16,<2.18\""
    ) from TENSORFLOW_IMPORT_ERROR

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 280)


In [ ]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_utils.experiment_config import build_default_config
from notebook_utils.feature_set_grid_builder import FeatureFrameBuilder, FeatureSetGridBuilder
from notebook_utils.metrics import ClassificationMetrics
from notebook_utils.model_report_builder import ModelReportBuilder
from notebook_utils.split_utils import make_split_dates, make_walk_forward_fold_specs, subset_by_dates

CONFIG = build_default_config(PROJECT_ROOT)

RNN_PARAM_GRID = [
    {
        "param_set": "gru_len5_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced",
        "sequence_length": 5,
        "cell_type": "GRU",
        "hidden_units": 8,
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 1e-3,
        "batch_size": 128,
        "max_epochs": 80,
        "patience": 8,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "gru_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced",
        "sequence_length": 10,
        "cell_type": "GRU",
        "hidden_units": 8,
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 1e-3,
        "batch_size": 128,
        "max_epochs": 80,
        "patience": 8,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "gru_len20_h16_drop0p3_l2_1e-3_lr5e-4_b128_balanced",
        "sequence_length": 20,
        "cell_type": "GRU",
        "hidden_units": 16,
        "dropout": 0.3,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 5e-4,
        "batch_size": 128,
        "max_epochs": 100,
        "patience": 10,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "lstm_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced",
        "sequence_length": 10,
        "cell_type": "LSTM",
        "hidden_units": 8,
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 1e-3,
        "batch_size": 128,
        "max_epochs": 80,
        "patience": 8,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b128_balanced",
        "sequence_length": 10,
        "cell_type": "GRU",
        "hidden_units": 16,
        "dropout": 0.3,
        "recurrent_dropout": 0.0,
        "dense_units": 8,
        "dense_dropout": 0.2,
        "l2": 1e-2,
        "learning_rate": 5e-4,
        "batch_size": 128,
        "max_epochs": 100,
        "patience": 10,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "lstm_len20_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b128_balanced",
        "sequence_length": 20,
        "cell_type": "LSTM",
        "hidden_units": 16,
        "dropout": 0.3,
        "recurrent_dropout": 0.0,
        "dense_units": 8,
        "dense_dropout": 0.2,
        "l2": 1e-2,
        "learning_rate": 5e-4,
        "batch_size": 128,
        "max_epochs": 100,
        "patience": 10,
        "class_weight_mode": "balanced",
    },
]

pd.DataFrame(RNN_PARAM_GRID)

feature_grid = FeatureSetGridBuilder.build(
    max_features_per_model=CONFIG["max_features_per_model"],
)

PRICE_FEATURES = feature_grid.price_features
VOLUME_FEATURE_OPTIONS = feature_grid.volume_feature_options
GDELT_FEATURE_OPTIONS = feature_grid.gdelt_feature_options
GDELT_SENTIMENT_FEATURE_OPTIONS = feature_grid.gdelt_sentiment_feature_options
GDELT_ATTENTION_FEATURE_OPTIONS = feature_grid.gdelt_attention_feature_options
REDDIT_FEATURE_OPTIONS = feature_grid.reddit_feature_options
REDDIT_ATTENTION_FEATURE_OPTIONS = feature_grid.reddit_attention_feature_options
GOOGLE_TRENDS_FEATURE_OPTIONS = feature_grid.google_trends_feature_options
GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS = feature_grid.google_score_attention_feature_options
DERIVED_FEATURE_COLUMNS = feature_grid.derived_feature_columns
BASE_VOLUME_OPTION = feature_grid.base_volume_option
BASELINE_FEATURE_SET = feature_grid.baseline_feature_set
FEATURE_SET_SPECS = feature_grid.feature_set_specs
FEATURE_SETS = feature_grid.feature_sets
FEATURE_SET_METADATA = feature_grid.feature_set_metadata
SKIPPED_FEATURE_SETS = feature_grid.skipped_feature_sets
FEATURE_SETS_TO_TEST = feature_grid.feature_sets_to_test
pd.DataFrame(RNN_PARAM_GRID)


In [ ]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = FeatureFrameBuilder.build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)
walk_forward_fold_specs = make_walk_forward_fold_specs(
    train_dates,
    n_folds=CONFIG["walk_forward_folds"],
    validation_size=CONFIG["walk_forward_validation_dates"],
    min_train_dates=CONFIG["walk_forward_min_train_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {
    "train": train_dates,
    "validation": validation_dates,
    "test": test_dates,
}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin(ClassificationMetrics.CLASS_VALUES)].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)
train_validation_df = subset_by_dates(modeled_df, list(train_dates) + list(validation_dates))

split_summary_rows = []
for split_name in ["train", "validation", "test"]:
    all_split_df = feature_df[feature_df["split"].eq(split_name)]
    modeled_split_df = modeled_df[modeled_df["split"].eq(split_name)]
    class_rates = modeled_split_df["target"].value_counts(normalize=True)
    split_summary_rows.append(
        {
            "split": split_name,
            "session_rows": len(all_split_df),
            "modeled_rows": len(modeled_split_df),
            "session_dates": all_split_df["date"].nunique(),
            "modeled_dates": modeled_split_df["date"].nunique(),
            "date_min": all_split_df["date"].min(),
            "date_max": all_split_df["date"].max(),
            "target_down_rate": float(class_rates.get(0, 0.0)),
            "target_neutral_rate": float(class_rates.get(1, 0.0)),
            "target_up_rate": float(class_rates.get(2, 0.0)),
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
walk_forward_fold_summary_df = pd.DataFrame(
    [
        {
            "fold": spec["fold"],
            "train_n_dates": spec["train_n_dates"],
            "train_date_min": spec["train_date_min"],
            "train_date_max": spec["train_date_max"],
            "validation_n_dates": spec["validation_n_dates"],
            "validation_date_min": spec["validation_date_min"],
            "validation_date_max": spec["validation_date_max"],
        }
        for spec in walk_forward_fold_specs
    ]
)

split_summary_df

In [ ]:
walk_forward_fold_summary_df


In [ ]:
neutral_summary_by_ticker_df = (
    feature_df.groupby("ticker")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reset_index()
)
neutral_summary_by_ticker_df["neutral_rate_among_available"] = (
    neutral_summary_by_ticker_df["neutral"] / neutral_summary_by_ticker_df["target_available"]
)
neutral_summary_by_ticker_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_ticker_df["neutral_rate_among_available"]

neutral_summary_by_split_df = (
    feature_df[feature_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
neutral_summary_by_split_df["neutral_rate_among_available"] = (
    neutral_summary_by_split_df["neutral"] / neutral_summary_by_split_df["target_available"]
)
neutral_summary_by_split_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_split_df["neutral_rate_among_available"]

print("Neutral coverage by split")
print(neutral_summary_by_split_df.to_string(index=False))
print("\nNeutral coverage by ticker")
neutral_summary_by_ticker_df


In [ ]:
missing_requested_feature_sets = [name for name in FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f"Unknown feature sets: {missing_requested_feature_sets}")

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(FEATURE_SETS[feature_set_name]),
            "features": FEATURE_SETS[feature_set_name],
        }
        for feature_set_name in FEATURE_SETS_TO_TEST
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

all_available_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(features),
            "features": features,
        }
        for feature_set_name, features in FEATURE_SETS.items()
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

skipped_feature_sets_df = pd.DataFrame(SKIPPED_FEATURE_SETS)

print(f"Selection metric: {CONFIG['selection_metric']}")
print(f"Primary validation metric: {CONFIG['primary_validation_metric']}")
print(f"Walk-forward folds: {len(walk_forward_fold_specs)}")
print(f"Target classes: {ClassificationMetrics.CLASS_LABELS}")
print("Prediction rule: multiclass argmax")
print(f"Max features per model: {CONFIG['max_features_per_model']}")
print(f"Available feature sets: {len(FEATURE_SETS)}")
print(f"RNN feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets to test: {sum('attention' in FEATURE_SET_METADATA[name]['feature_family'] for name in FEATURE_SETS_TO_TEST)}")
print(f"Skipped feature sets above max feature limit: {len(SKIPPED_FEATURE_SETS)}")
print(f"RNN parameter sets: {len(RNN_PARAM_GRID)}")
print(f"Walk-forward validation fits: {len(FEATURE_SETS_TO_TEST) * len(RNN_PARAM_GRID) * len(walk_forward_fold_specs)}")

candidate_feature_sets_df


In [ ]:
from __future__ import annotations


RNN_PARAM_COLUMNS = [
    "sequence_length",
    "cell_type",
    "hidden_units",
    "dropout",
    "recurrent_dropout",
    "dense_units",
    "dense_dropout",
    "l2",
    "learning_rate",
    "batch_size",
    "max_epochs",
    "patience",
    "class_weight_mode",
]


class SequencePreprocessor:
    def fit(self, X: np.ndarray) -> "SequencePreprocessor":
        flat = X.reshape(-1, X.shape[-1]).astype("float32", copy=False)
        self.median_ = np.nanmedian(flat, axis=0)
        self.median_ = np.where(np.isfinite(self.median_), self.median_, 0.0).astype("float32")
        filled = np.where(np.isnan(flat), self.median_, flat)
        self.mean_ = filled.mean(axis=0).astype("float32")
        self.std_ = filled.std(axis=0)
        self.std_ = np.where((self.std_ > 0.0) & np.isfinite(self.std_), self.std_, 1.0).astype("float32")
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        filled = np.where(np.isnan(X), self.median_, X)
        scaled = (filled - self.mean_) / self.std_
        return scaled.astype("float32")

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)


def prepare_train_eval_feature_frames(
    features: list[str],
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "google_trends_above_ticker_train_median" in features:
        return FeatureFrameBuilder.add_google_trends_train_median_feature(train_input_df, eval_input_df)
    return train_input_df, eval_input_df


def prepare_sequence_source_feature_frame(
    features: list[str],
    train_input_df: pd.DataFrame,
    sequence_source_df: pd.DataFrame,
) -> pd.DataFrame:
    if "google_trends_above_ticker_train_median" in features:
        _, sequence_source_with_flag_df = FeatureFrameBuilder.add_google_trends_train_median_feature(
            train_input_df,
            sequence_source_df,
        )
        return sequence_source_with_flag_df
    return sequence_source_df


def params_from_result_row(row: dict | pd.Series) -> dict:
    return {
        "param_set": row["param_set"],
        "sequence_length": int(row["sequence_length"]),
        "cell_type": row["cell_type"],
        "hidden_units": int(row["hidden_units"]),
        "dropout": float(row["dropout"]),
        "recurrent_dropout": float(row["recurrent_dropout"]),
        "dense_units": int(row["dense_units"]),
        "dense_dropout": float(row["dense_dropout"]),
        "l2": float(row["l2"]),
        "learning_rate": float(row["learning_rate"]),
        "batch_size": int(row["batch_size"]),
        "max_epochs": int(row["max_epochs"]),
        "patience": int(row["patience"]),
        "class_weight_mode": row.get("class_weight_mode", "balanced"),
    }


def class_weight_from_target(y_train: np.ndarray, mode: str | None) -> dict[int, float] | None:
    if mode in [None, "none"]:
        return None
    if mode != "balanced":
        raise ValueError(f"Unsupported class_weight_mode: {mode}")
    counts = pd.Series(y_train).value_counts()
    n_total = float(len(y_train))
    n_classes = float(len(counts))
    if n_classes <= 0.0:
        return None
    return {
        int(class_value): n_total / (n_classes * float(class_count))
        for class_value, class_count in counts.items()
        if class_count > 0
    }


def split_train_for_early_stopping(train_input_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    dates = np.array(sorted(train_input_df["date"].unique()))
    n_dates = len(dates)
    n_validation = max(
        CONFIG["min_early_stopping_dates"],
        int(round(n_dates * CONFIG["early_stopping_fraction"])),
    )
    n_validation = min(n_validation, max(1, n_dates - CONFIG["min_fit_dates"]))
    fit_dates = dates[:-n_validation]
    early_stop_dates = dates[-n_validation:]
    return subset_by_dates(train_input_df, fit_dates), subset_by_dates(train_input_df, early_stop_dates)


def make_sequence_dataset(
    sequence_source_df: pd.DataFrame,
    sample_df: pd.DataFrame,
    features: list[str],
    sequence_length: int,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    source_lookup = {}
    for ticker, ticker_df in sequence_source_df.sort_values(["ticker", "date"]).groupby("ticker", sort=False):
        dates = [pd.Timestamp(value) for value in ticker_df["date"]]
        source_lookup[ticker] = {
            "positions": {date: position for position, date in enumerate(dates)},
            "values": ticker_df[features].to_numpy(dtype="float32"),
        }

    sequences = []
    targets = []
    metadata_rows = []
    for row in sample_df.sort_values(["date", "ticker"]).itertuples(index=False):
        ticker_data = source_lookup.get(row.ticker)
        if ticker_data is None:
            continue
        position = ticker_data["positions"].get(pd.Timestamp(row.date))
        if position is None:
            continue
        start = position - sequence_length + 1
        if start < 0:
            continue
        sequences.append(ticker_data["values"][start : position + 1])
        targets.append(int(row.target))
        metadata_rows.append({"date": row.date, "ticker": row.ticker, "target": int(row.target)})

    if not sequences:
        raise ValueError(f"No sequences were built for sequence_length={sequence_length}")
    return np.stack(sequences).astype("float32", copy=False), np.asarray(targets, dtype=int), pd.DataFrame(metadata_rows)


def build_rnn_model(params: dict, n_features: int) -> keras.Model:
    if tf is None:
        raise ImportError(
            "TensorFlow is not installed in this Python environment. Install TensorFlow before running the RNN notebook."
        ) from TENSORFLOW_IMPORT_ERROR

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(CONFIG["random_state"])
    regularizer = regularizers.l2(params["l2"]) if params["l2"] > 0 else None
    inputs = keras.Input(shape=(params["sequence_length"], n_features), dtype="float32")
    cell_kwargs = {
        "units": params["hidden_units"],
        "dropout": params["dropout"],
        "recurrent_dropout": params["recurrent_dropout"],
        "kernel_regularizer": regularizer,
        "recurrent_regularizer": regularizer,
    }
    if params["cell_type"] == "GRU":
        x = layers.GRU(**cell_kwargs)(inputs)
    elif params["cell_type"] == "LSTM":
        x = layers.LSTM(**cell_kwargs)(inputs)
    else:
        raise ValueError(f"Unsupported recurrent cell type: {params['cell_type']}")

    if params["dense_units"] > 0:
        x = layers.Dense(params["dense_units"], activation="relu", kernel_regularizer=regularizer)(x)
        if params["dense_dropout"] > 0:
            x = layers.Dropout(params["dense_dropout"])(x)

    outputs = layers.Dense(len(ClassificationMetrics.CLASS_VALUES), activation="softmax")(x)
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss="sparse_categorical_crossentropy",
    )
    return model



def add_param_columns(row: dict, params: dict, param_columns: list[str]) -> None:
    for column in param_columns:
        row[column] = params.get(column)


def evaluate_rnn_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    train_features_df, eval_features_df = prepare_train_eval_feature_frames(
        features,
        train_input_df,
        eval_input_df,
    )
    fit_input_df, early_stop_input_df = split_train_for_early_stopping(train_features_df)
    sequence_source_df = prepare_sequence_source_feature_frame(features, train_input_df, feature_df)

    sequence_length = params["sequence_length"]
    X_fit_raw, y_fit, _ = make_sequence_dataset(sequence_source_df, fit_input_df, features, sequence_length)
    X_early_raw, y_early, _ = make_sequence_dataset(sequence_source_df, early_stop_input_df, features, sequence_length)
    X_eval_raw, y_eval, eval_metadata_df = make_sequence_dataset(sequence_source_df, eval_features_df, features, sequence_length)

    preprocessor = SequencePreprocessor()
    X_fit = preprocessor.fit_transform(X_fit_raw)
    X_early = preprocessor.transform(X_early_raw)
    X_eval = preprocessor.transform(X_eval_raw)

    model = build_rnn_model(params, n_features=len(features))
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=params["patience"],
            restore_best_weights=True,
        )
    ]
    history = model.fit(
        X_fit,
        y_fit,
        validation_data=(X_early, y_early),
        epochs=params["max_epochs"],
        batch_size=params["batch_size"],
        verbose=0,
        callbacks=callbacks,
        class_weight=class_weight_from_target(y_fit, params.get("class_weight_mode")),
    )
    probabilities = model(X_eval, training=False).numpy().astype("float32", copy=False)
    preds = ClassificationMetrics.predictions_from_probabilities(probabilities)

    del model, history, callbacks, X_fit, X_early, X_eval, X_fit_raw, X_early_raw, X_eval_raw
    keras.backend.clear_session()
    gc.collect()

    metric_result = ClassificationMetrics.metrics_from_predictions(y_eval, preds)
    preds = metric_result.pop("preds")
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    row = {
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        **metric_result,
    }
    add_param_columns(row, params, RNN_PARAM_COLUMNS)

    if not return_predictions:
        return row

    predictions_df = eval_metadata_df.copy()
    for column, values in ClassificationMetrics.probability_column_dict(probabilities).items():
        predictions_df[column] = values
    predictions_df["prediction"] = preds
    return row, predictions_df


def evaluate_rnn_config_walk_forward(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    modeled_input_df: pd.DataFrame,
    fold_specs: list[dict],
) -> tuple[dict, list[dict]]:
    fold_rows = []
    for spec in fold_specs:
        fold_row = evaluate_rnn_params(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            train_input_df=subset_by_dates(modeled_input_df, spec["train_dates"]),
            eval_input_df=subset_by_dates(modeled_input_df, spec["validation_dates"]),
            split_name="walk_forward_validation",
        )
        fold_rows.append(
            {
                **fold_row,
                "fold": spec["fold"],
                "fold_train_n_dates": spec["train_n_dates"],
                "fold_validation_n_dates": spec["validation_n_dates"],
                "fold_train_date_min": spec["train_date_min"],
                "fold_train_date_max": spec["train_date_max"],
                "fold_validation_date_min": spec["validation_date_min"],
                "fold_validation_date_max": spec["validation_date_max"],
            }
        )

    fold_results_df = pd.DataFrame(fold_rows)
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    primary_metric = CONFIG["primary_validation_metric"]
    metric_mean = float(fold_results_df[primary_metric].mean())
    summary_row = {
        "split": "walk_forward_validation",
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        "balanced_accuracy": metric_mean,
        "accuracy": float(fold_results_df["accuracy"].mean()),
        "f1_score": float(fold_results_df["f1_score"].mean()),
        "f1_weighted": float(fold_results_df["f1_weighted"].mean()),
    }
    add_param_columns(summary_row, params, RNN_PARAM_COLUMNS)
    return summary_row, fold_rows



In [ ]:
selection_metric = CONFIG["selection_metric"]
walk_forward_grid_rows = []
walk_forward_fold_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in RNN_PARAM_GRID:
        summary_row, fold_rows = evaluate_rnn_config_walk_forward(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            modeled_input_df=modeled_df,
            fold_specs=walk_forward_fold_specs,
        )
        walk_forward_grid_rows.append(summary_row)
        walk_forward_fold_rows.extend(fold_rows)

walk_forward_grid_results_df = pd.DataFrame(walk_forward_grid_rows)
walk_forward_fold_results_df = pd.DataFrame(walk_forward_fold_rows)
if selection_metric not in walk_forward_grid_results_df.columns:
    raise KeyError(f"Selection metric is not available: {selection_metric}")

validation_grid_results_df = walk_forward_grid_results_df.sort_values(
    [selection_metric, "balanced_accuracy", "f1_score", "accuracy", "feature_set", "param_set"],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)


In [ ]:
validation_best_by_feature_set_df = ModelReportBuilder.select_best_validation_by_feature_set(
    validation_grid_results_df,
    selection_metric=CONFIG["selection_metric"],
)

validation_best_by_feature_set_report_df = ModelReportBuilder.build_validation_best_by_feature_set_report(
    validation_best_by_feature_set_df,
    param_columns=RNN_PARAM_COLUMNS,
)

validation_best_by_feature_set_report_df


In [ ]:
best_validation_params_df = validation_best_by_feature_set_df.copy()


In [ ]:
validation_refit_rows = []
test_rows = []

for row in best_validation_params_df.to_dict(orient="records"):
    params = params_from_result_row(row)
    feature_set_name = row["feature_set"]
    validation_refit_row = evaluate_rnn_params(
        feature_set_name=feature_set_name,
        features=FEATURE_SETS[feature_set_name],
        params=params,
        train_input_df=train_df,
        eval_input_df=validation_df,
        split_name="validation_refit_train",
    )
    validation_refit_rows.append(validation_refit_row)
    test_rows.append(
        evaluate_rnn_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_validation_df,
            eval_input_df=test_df,
            split_name="test_refit_train_validation",
        )
    )

validation_refit_results_df = pd.DataFrame(validation_refit_rows)
test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ["balanced_accuracy", "f1_score", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

(
    simple_hyperparameter_summary_df,
    baseline_walk_forward_row,
    baseline_validation_refit_row,
    baseline_test_row,
) = ModelReportBuilder.build_simple_hyperparameter_summary(
    best_validation_params_df=best_validation_params_df,
    validation_refit_results_df=validation_refit_results_df,
    test_best_validation_params_df=test_best_validation_params_df,
    baseline_feature_set=BASELINE_FEATURE_SET,
)

In [ ]:
validation_selected_family_test_report_df = ModelReportBuilder.build_validation_selected_family_test_report(
    simple_hyperparameter_summary_df,
    param_columns=RNN_PARAM_COLUMNS,
)

final_test_verification_path = ModelReportBuilder.save_final_test_verification(
    validation_selected_family_test_report_df,
    model_name="rnn",
    output_dir=PROJECT_ROOT / "notebooks" / "outputs",
)
print(f"Saved final test verification to: {final_test_verification_path}")

validation_selected_family_test_report_df
